In [ ]:
# Example 1: Verify the labextension + server-extension discovery hooks.
# Jupyter Server / JupyterLab call these by name on package import.
import jupyter_ai_chat_commands as jacc

lab_paths = jacc._jupyter_labextension_paths()
assert isinstance(lab_paths, list) and len(lab_paths) == 1
assert lab_paths[0]["src"] == "labextension"
assert lab_paths[0]["dest"] == "@jupyter-ai/chat-commands"
print("labextension paths:", lab_paths)

server_points = jacc._jupyter_server_extension_points()
assert isinstance(server_points, list) and len(server_points) == 1
assert server_points[0]["module"] == "jupyter_ai_chat_commands"
# The 'app' entry should be the ExtensionApp class.
from jupyter_ai_chat_commands.extension_app import JaiChatCommandsExtension
assert server_points[0]["app"] is JaiChatCommandsExtension
print("server extension app:", server_points[0]["app"].__name__)


In [ ]:
# Example 2: Inspect the JaiChatCommandsExtension class shape without instantiating
# (instantiating an ExtensionApp requires a live jupyter_server).
from jupyter_ai_chat_commands.extension_app import JaiChatCommandsExtension
from jupyter_server.extension.application import ExtensionApp

assert issubclass(JaiChatCommandsExtension, ExtensionApp)
assert JaiChatCommandsExtension.name == "jupyter_ai_chat_commands"

# It registers exactly one tornado route under the extension's URL namespace.
assert isinstance(JaiChatCommandsExtension.handlers, list)
assert len(JaiChatCommandsExtension.handlers) == 1
route_pattern, handler_cls = JaiChatCommandsExtension.handlers[0]
assert "jupyter-ai-chat-commands/get-example" in route_pattern
print("route pattern:", route_pattern)
print("handler class:", handler_cls.__name__)

# Slash-command/observer methods must be present on the class.
for method in ("initialize_settings", "initialize_async", "on_chat_init", "on_slash_command", "handle_refresh_personas"):
    assert callable(getattr(JaiChatCommandsExtension, method)), method
print("all expected lifecycle methods present")


In [ ]:
# Example 3: get_first_word() utility — exercise edge cases.
from jupyter_ai_chat_commands.extension_app import get_first_word

cases = [
    ("hello world", "hello"),
    ("   leading whitespace ok", "leading"),
    ("/refresh-personas", "/refresh-personas"),
    ("\tword\nrest", "word"),
    ("oneword", "oneword"),
    ("", None),
    ("   ", None),
    ("\n\t  ", None),
]
for input_str, expected in cases:
    got = get_first_word(input_str)
    assert got == expected, f"get_first_word({input_str!r}) -> {got!r}, expected {expected!r}"
    print(f"  {input_str!r:40s} -> {got!r}")
print("\nget_first_word: all edge cases pass")


In [ ]:
# Example 4: RouteHandler is a properly-shaped Jupyter Server APIHandler.
from jupyter_ai_chat_commands.handlers import RouteHandler, setup_handlers
from jupyter_server.base.handlers import APIHandler
import inspect

assert issubclass(RouteHandler, APIHandler)
assert callable(getattr(RouteHandler, "get", None))

# The GET method must be decorated with @tornado.web.authenticated.
# That decorator stashes _wrapped or similar; the simplest check is
# that the method's qualname / source mentions tornado.web.authenticated.
src = inspect.getsource(RouteHandler.get)
assert "authenticated" in src
print("RouteHandler.get is @tornado.web.authenticated")

# setup_handlers should be a function expecting a web_app argument.
sig = inspect.signature(setup_handlers)
assert list(sig.parameters) == ["web_app"]
print("setup_handlers signature:", sig)


In [ ]:
# Example 5: setup_handlers wires the route into a mock web_app correctly.
from jupyter_ai_chat_commands.handlers import setup_handlers, RouteHandler

class _MockWebApp:
    def __init__(self, base_url="/"):
        self.settings = {"base_url": base_url}
        self.added = []
    def add_handlers(self, host_pattern, handlers):
        self.added.append((host_pattern, handlers))

app = _MockWebApp(base_url="/lab/")
setup_handlers(app)

assert len(app.added) == 1
host_pattern, handlers = app.added[0]
assert host_pattern == ".*$"
assert len(handlers) == 1
route_pattern, cls = handlers[0]
assert cls is RouteHandler
assert "jupyter-ai-chat-commands/get-example" in route_pattern
assert route_pattern.startswith("/lab/")
print("setup_handlers registered:", route_pattern, "->", cls.__name__)
assert True, "jupyter_ai_chat_commands public API exercised end-to-end"
